# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the FAIR² dataset using the `mlcroissant` library. We'll walk through loading the schema, examining the available record sets and fields (always referencing entities by their `@id`), extracting data, basic analyses, and plotting.

### Dataset Source
The dataset source is provided via the following Croissant schema URL:
- https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json

In [ ]:
# Install the mlcroissant library
!pip install -U mlcroissant

## 1. Data Loading
We load metadata and records from the dataset using `mlcroissant` and inspect the dataset details.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata via Croissant
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset name: {metadata.name}\n")
print(f"Dataset description: {metadata.description}\n")
print(f"Croissant version: {getattr(metadata, 'conformsTo', 'N/A')}")
print(f"License: {getattr(metadata, 'license', 'N/A')}")

# Show available top-level attributes
print("\nTop-level metadata fields:")
print([attr for attr in dir(metadata) if not attr.startswith('_') and not callable(getattr(metadata, attr))])

## 2. Data Overview
Explore the available record sets and their fields, listing their `@id`s. In Croissant, each record set, field, and column has a unique `@id`. We use these for all further references.

In [ ]:
# List all record sets and their fields by @id
print("Available Record Sets (by @id):\n")

record_sets = getattr(metadata, 'recordSet', [])
if not record_sets:
    print("No record sets found in the metadata.\nTry accessing 'dataset_files' instead.")
else:
    for rs in record_sets:
        print(f"- RecordSet @id: {getattr(rs, '@id', 'N/A')}")
        fields = getattr(rs, 'field', [])
        if fields:
            print("  Fields (by @id):")
            for f in fields:
                print(f"    * {getattr(f, '@id', f)} (name: {getattr(f, 'name', '')})")
        else:
            print("  No fields defined on this RecordSet.")
        print()

# If no record sets found, try auto-detect the available ones using the dataset.records() API
if not record_sets:
    print("Attempting to list record_set IDs registered in the dataset.records() API.\n")
    available_ids = dataset.record_sets()
    for rid in available_ids:
        print(f"* RecordSet @id: {rid}")

## 3. Data Extraction
We'll now load data for one or more record sets. All dataset entities are referenced by their `@id`. Please inspect the "Data Overview" output above and pick the main record set for extraction.

In [ ]:
# List record sets available
record_sets_ids = dataset.record_sets()
print(f"Available record sets @id: {record_sets_ids}")

# For mass spectrometry datasets, the first record set is usually the table of protein or peptide records.
main_record_set_id = record_sets_ids[0] if len(record_sets_ids) else None
print(f"Selected main record set for extraction: {main_record_set_id}")

dataframes = {}
if main_record_set_id:
    records = list(dataset.records(record_set=main_record_set_id))
    df = pd.DataFrame(records)
    dataframes[main_record_set_id] = df
    print(f"Extracted columns (@id): {df.columns.tolist()}")
    display(df.head())
else:
    print("No suitable record set found to extract records from.")

## 4. Exploratory Data Analysis (EDA)
Let's perform some simple EDA steps. This includes filtering, normalization, and grouping. We select a numeric field (e.g. peptide count, coverage, molecular weight) by its `@id` and show example operations.

In [ ]:
import numpy as np

# Inspect columns and select a numeric field (use @id or column name as shown above)
df = dataframes.get(main_record_set_id, None)
if df is not None:
    print("Available columns in main DataFrame:")
    print(df.columns.tolist())

    # Guess at possible numeric field ids - user should replace with the exact @id as needed
    candidate_numeric_fields = [col for col in df.columns if any(word in col.lower() for word in ['count', 'abundance', 'coverage', 'weight', 'mw', 'number', 'peptide'])]
    numeric_field = candidate_numeric_fields[0] if candidate_numeric_fields else df.select_dtypes('number').columns[0]
    print(f"Using numeric field: {numeric_field}")

    threshold = df[numeric_field].mean() if np.issubdtype(df[numeric_field].dtype, np.number) else 10
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold:.2f} (top 5):")
    display(filtered_df.head())

    # Normalization
    colname_norm = f"{numeric_field}_normalized"
    filtered_df[colname_norm] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"First 5 records for normalized {numeric_field}:")
    display(filtered_df[[numeric_field, colname_norm]].head())

    # Grouping by another field - search for plausible group field
    candidate_group_fields = [col for col in df.columns if any(word in col.lower() for word in ['sample', 'type', 'group', 'category', 'organism'])]
    group_field = candidate_group_fields[0] if candidate_group_fields else df.columns[0]

    if group_field in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"Mean {numeric_field} grouped by {group_field}:")
        display(grouped_df.head())
else:
    print("No data available for EDA.")

## 5. Visualization
We'll visualize the distribution of the selected numeric field, and perhaps a scatter plot of two numeric attributes if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if df is not None:
    # Histogram of main numeric field
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field], kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()

    # Try finding a second numeric field for scatter plot
    num_fields = df.select_dtypes('number').columns.tolist()
    second_field = [col for col in num_fields if col != numeric_field]
    if second_field:
        plt.figure(figsize=(7, 5))
        sns.scatterplot(x=df[numeric_field], y=df[second_field[0]])
        plt.xlabel(numeric_field)
        plt.ylabel(second_field[0])
        plt.title(f"{numeric_field} vs. {second_field[0]}")
        plt.show()
else:
    print("No data to plot.")

## 6. Conclusion
- We explored the FAIR² mass spectrometry dataset using the Croissant schema and `mlcroissant` library.
- All entities (record sets, fields, columns) were referenced by their `@id`.
- Data was loaded into Pandas DataFrames and filtered/normalized for exploratory analysis.
- Visualizations revealed the distribution and simple relationships within the dataset.

**Next steps:**
- Explore annotation metadata, variance across different samples, and link to any provided documentation/sections within the Croissant metadata.
- For advanced analysis, apply machine learning preprocessing or statistical tests using the field `@id`s as references throughout your code.